In [1]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from matplotlib.colors import TwoSlopeNorm

# === PUBLICATION-QUALITY STYLING (HIGHEST STANDARD) ===
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 10
rcParams['axes.labelsize'] = 10
rcParams['axes.titlesize'] = 11
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['legend.fontsize'] = 9
rcParams['figure.dpi'] = 300
rcParams['savefig.dpi'] = 300
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
rcParams['axes.linewidth'] = 0.8
rcParams['xtick.major.width'] = 0.8
rcParams['ytick.major.width'] = 0.8

# === DATA INPUT ===
# Comparison pairs with agent names
comparisons = [
    'Aggressive vs Conservative',
    'Aggressive vs Balanced',
    'Aggressive vs Opportunistic',
    'Conservative vs Balanced',
    'Conservative vs Opportunistic',
    'Balanced vs Opportunistic'
]

# Metrics
metrics = ['Win', 'Economic', 'Jhyap', 'Cards', 'Risk']

# Cohen's d effect sizes (rows: comparisons, columns: metrics)
d_values = np.array([
    [0.35647654277892116, 0.24304704842994962, 0.7038154864264691, -0.01454401223559176, -0.44566840549086034],
    [0.21849658401458724, 0.16784579203851224, 0.4064717718667322, -0.01720179527303693, -0.10645418642597129],
    [0.3316709551793689, 0.28601119450533996, 0.41602940784694165, -0.07404753579747994, 0.09365007938783505],
    [-0.1364622779828218, -0.07528599474941304, -0.28675996358828765, -0.0028201200057098685, 0.3726170550977614],
    [-0.02445177805887847, 0.0594499615068536, -0.27745404377779787, -0.059431031396931566, 0.5535343544704581],
    [0.11198368267771919, 0.12834800943621139, 0.009289497949557576, -0.05602380895309224, 0.2008443086549855]
])

# P-values (rows: comparisons, columns: metrics)
p_values = np.array([
    [1.2839639276787884e-15, 4.4002100058366736e-08, 1.927659021530055e-53, 0.7422402618516641, 1.2590603244307342e-08],
    [8.384034291024766e-07, 0.0001514520795905967, 9.3093057975167e-20, 0.6972856481389622, 0.18056654657674584],
    [9.473733258922113e-14, 1.250319520269698e-10, 1.3283134304311435e-20, 0.09415172044166707, 0.2586824201693244],
    [0.0020541568778487587, 0.08877952213279189, 1.12060393753114e-10, 0.949150949519078, 0.0001293045842027969],
    [0.5803167549175454, 0.1789231341127722, 4.3028632412066967e-10, 0.17906229155149203, 9.405565508505237e-09],
    [0.011394522797131918, 0.0037385319270242137, 0.8336146944860158, 0.2052807765891998, 0.030514562511972546]
])

def get_significance_marker(p_value):
    """
    Return significance marker based on p-value.

    Parameters:
    -----------
    p_value : float
        The p-value to evaluate

    Returns:
    --------
    str : Significance marker ('***', '**', '*', or '')
    """
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

def create_comparison_heatmap(output_file='player_comparison_heatmap.png'):
    """
    Create heatmap visualization for pairwise player comparisons.

    The heatmap shows:
    - Cohen's d effect sizes as cell colors
    - Statistical significance as asterisks
    - Diverging colormap centered at 0

    Parameters:
    -----------
    output_file : str
        Output filename for the figure
    """

    # === CREATE FIGURE ===
    fig, ax = plt.subplots(figsize=(8, 5))

    # === CREATE COLORMAP (Diverging, centered at 0) ===
    # Find absolute maximum for symmetric scale
    vmax = max(abs(np.min(d_values)), abs(np.max(d_values)))
    vmin = -vmax

    # Create diverging normalization centered at 0
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # === PLOT HEATMAP ===
    im = ax.imshow(d_values, cmap='RdBu_r', aspect='auto', norm=norm)

    # === ADD COLORBAR ===
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cohen's d Effect Size", rotation=270, labelpad=20, fontweight='medium')
    cbar.outline.set_linewidth(0.8)

    # === CONFIGURE AXES ===
    ax.set_xticks(np.arange(len(metrics)))
    ax.set_yticks(np.arange(len(comparisons)))
    ax.set_xticklabels(metrics, fontweight='medium')
    ax.set_yticklabels(comparisons, fontweight='medium')

    # Rotate x-axis labels for better readability
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

    # === ADD GRID LINES ===
    # Minor ticks for grid between cells
    ax.set_xticks(np.arange(len(metrics)) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(comparisons)) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
    ax.tick_params(which="minor", size=0)

    # === ANNOTATE CELLS WITH VALUES AND SIGNIFICANCE ===
    for i in range(len(comparisons)):
        for j in range(len(metrics)):
            d_val = d_values[i, j]
            p_val = p_values[i, j]
            sig_marker = get_significance_marker(p_val)

            # Format d value
            d_text = f'{d_val:.3f}'

            # Determine text color (white for strong colors, black for weak)
            if abs(d_val) > vmax * 0.5:
                text_color = 'white'
            else:
                text_color = 'black'

            # Add d value
            ax.text(j, i, d_text,
                   ha="center", va="center",
                   color=text_color, fontsize=9,
                   fontweight='bold')

            # Add significance marker below d value
            if sig_marker:
                ax.text(j, i + 0.35, sig_marker,
                       ha="center", va="center",
                       color=text_color, fontsize=10,
                       fontweight='bold')

    # === LABELS AND TITLE ===
    ax.set_xlabel('Performance Metrics', fontweight='medium', fontsize=10)
    ax.set_ylabel('Agent Comparisons', fontweight='medium', fontsize=10)
    ax.set_title('Pairwise Agent Performance Comparison: Effect Sizes and Statistical Significance',
                fontweight='bold', fontsize=11, pad=15)

    # === ADD LEGEND FOR SIGNIFICANCE ===
    legend_text = '* p < 0.05   ** p < 0.01   *** p < 0.001'
    fig.text(0.5, 0.02, legend_text, ha='center', fontsize=9,
            style='italic', color='#555555',
            bbox=dict(boxstyle='round', facecolor='white', 
                     edgecolor='gray', alpha=0.8, linewidth=0.8))

    # === STYLING REFINEMENTS ===
    ax.spines['top'].set_linewidth(0.8)
    ax.spines['right'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.spines['left'].set_linewidth(0.8)

    # === SAVE FIGURE ===
    plt.tight_layout(rect=[0, 0.04, 1, 1])  # Leave space for legend at bottom
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_file}")

    # === PRINT STATISTICS ===
    print("\n" + "="*80)
    print("EFFECT SIZE SUMMARY")
    print("="*80)

    print(f"\nOverall Statistics:")
    print(f"  Maximum effect size: {np.max(d_values):.3f}")
    print(f"  Minimum effect size: {np.min(d_values):.3f}")
    print(f"  Mean absolute effect size: {np.mean(np.abs(d_values)):.3f}")

    print(f"\nSignificant Effects (p < 0.05):")
    significant_count = np.sum(p_values < 0.05)
    total_comparisons = p_values.size
    print(f"  {significant_count}/{total_comparisons} comparisons ({100*significant_count/total_comparisons:.1f}%)")

    print(f"\nLargest Effect Sizes by Metric:")
    for j, metric in enumerate(metrics):
        max_idx = np.argmax(np.abs(d_values[:, j]))
        max_val = d_values[max_idx, j]
        max_comp = comparisons[max_idx]
        print(f"  {metric}: {max_val:.3f} ({max_comp})")

    print("\n" + "="*80)

    plt.close()
    return True

def main():
    """Main execution function."""
    print("="*80)
    print("PAIRWISE AGENT COMPARISON HEATMAP - PUBLICATION QUALITY")
    print("="*80)

    print("\nGenerating heatmap visualization...")
    success = create_comparison_heatmap()

    if success:
        print("\n" + "="*80)
        print("✓ SUCCESS: Heatmap generated!")
        print("="*80)
        print("\nVisualization features:")
        print("  • Diverging colormap (red = positive, blue = negative)")
        print("  • Cohen's d effect sizes displayed in cells")
        print("  • Statistical significance markers (*, **, ***)")
        print("  • Publication-ready formatting (300 DPI)")
        print("\nOutput: player_comparison_heatmap.png")
        print("="*80)
    else:
        print("\n❌ FAILED: Could not generate heatmap")

if __name__ == "__main__":
    main()

PAIRWISE AGENT COMPARISON HEATMAP - PUBLICATION QUALITY

Generating heatmap visualization...
✓ Saved: player_comparison_heatmap.png

EFFECT SIZE SUMMARY

Overall Statistics:
  Maximum effect size: 0.704
  Minimum effect size: -0.446
  Mean absolute effect size: 0.208

Significant Effects (p < 0.05):
  18/30 comparisons (60.0%)

Largest Effect Sizes by Metric:
  Win: 0.356 (Aggressive vs Conservative)
  Economic: 0.286 (Aggressive vs Opportunistic)
  Jhyap: 0.704 (Aggressive vs Conservative)
  Cards: -0.074 (Aggressive vs Opportunistic)
  Risk: 0.554 (Conservative vs Opportunistic)


✓ SUCCESS: Heatmap generated!

Visualization features:
  • Diverging colormap (red = positive, blue = negative)
  • Cohen's d effect sizes displayed in cells
  • Statistical significance markers (*, **, ***)
  • Publication-ready formatting (300 DPI)

Output: player_comparison_heatmap.png


In [2]:
# ============================================================================
# FIGURE: Championship Tournament Pairwise Comparison Heatmap
# Filename: create_championship_heatmap.py
# Description: Heatmap showing Cohen's d effect sizes for tournament comparisons
# For academic research paper use
# ============================================================================

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from matplotlib.colors import TwoSlopeNorm

# === PUBLICATION-QUALITY STYLING (HIGHEST STANDARD) ===
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 10
rcParams['axes.labelsize'] = 10
rcParams['axes.titlesize'] = 11
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['legend.fontsize'] = 9
rcParams['figure.dpi'] = 300
rcParams['savefig.dpi'] = 300
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
rcParams['axes.linewidth'] = 0.8
rcParams['xtick.major.width'] = 0.8
rcParams['ytick.major.width'] = 0.8

# === DATA INPUT ===
# Comparison pairs
comparisons = [
    'Aggressive vs ISMCTS',
    'Aggressive vs PPO',
    'Aggressive vs Random',
    'ISMCTS vs PPO',
    'ISMCTS vs Random',
    'PPO vs Random'
]

# Metrics
metrics = ['Win', 'Economic', 'Jhyap']

# Cohen's d effect sizes (rows: comparisons, columns: metrics)
d_values = np.array([
    [2.606, 2.073, 5.022],
    [3.576, 2.759, 5.321],
    [3.613, 2.740, 5.567],
    [0.343, 0.682, 0.050],
    [0.355, 0.639, 0.092],
    [0.017, -0.076, 0.043]
])

# P-values (rows: comparisons, columns: metrics)
p_values = np.array([
    [0.0000, 0.0000, 0.0000],
    [0.0000, 0.0000, 0.0000],
    [0.0000, 0.0000, 0.0000],
    [0.0000, 0.0000, 0.2580],
    [0.0000, 0.0000, 0.0380],
    [0.7037, 0.0878, 0.3330]
])

def get_significance_marker(p_value):
    """
    Return significance marker based on p-value.

    Parameters:
    -----------
    p_value : float
        The p-value to evaluate

    Returns:
    --------
    str : Significance marker ('***', '**', '*', or '')
    """
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

def create_championship_heatmap(output_file='championship_comparison_heatmap.png'):
    """
    Create heatmap visualization for championship tournament comparisons.

    The heatmap shows:
    - Cohen's d effect sizes as cell colors
    - Statistical significance as asterisks
    - Diverging colormap centered at 0

    Parameters:
    -----------
    output_file : str
        Output filename for the figure
    """

    # === CREATE FIGURE ===
    fig, ax = plt.subplots(figsize=(6.5, 6))

    # === CREATE COLORMAP (Diverging, centered at 0) ===
    # Find absolute maximum for symmetric scale
    vmax = max(abs(np.min(d_values)), abs(np.max(d_values)))
    vmin = -vmax

    # Create diverging normalization centered at 0
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # === PLOT HEATMAP ===
    im = ax.imshow(d_values, cmap='RdBu_r', aspect='auto', norm=norm)

    # === ADD COLORBAR ===
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cohen's d Effect Size", rotation=270, labelpad=20, fontweight='medium')
    cbar.outline.set_linewidth(0.8)

    # === CONFIGURE AXES ===
    ax.set_xticks(np.arange(len(metrics)))
    ax.set_yticks(np.arange(len(comparisons)))
    ax.set_xticklabels(metrics, fontweight='medium')
    ax.set_yticklabels(comparisons, fontweight='medium')

    # Rotate x-axis labels for better readability
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

    # === ADD GRID LINES ===
    # Minor ticks for grid between cells
    ax.set_xticks(np.arange(len(metrics)) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(comparisons)) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=2.5)
    ax.tick_params(which="minor", size=0)

    # === ANNOTATE CELLS WITH VALUES AND SIGNIFICANCE ===
    for i in range(len(comparisons)):
        for j in range(len(metrics)):
            d_val = d_values[i, j]
            p_val = p_values[i, j]
            sig_marker = get_significance_marker(p_val)

            # Format d value
            d_text = f'{d_val:.3f}'

            # Determine text color (white for strong colors, black for weak)
            if abs(d_val) > vmax * 0.4:
                text_color = 'white'
            else:
                text_color = 'black'

            # Add d value
            ax.text(j, i, d_text,
                   ha="center", va="center",
                   color=text_color, fontsize=10,
                   fontweight='bold')

            # Add significance marker below d value
            if sig_marker:
                ax.text(j, i + 0.35, sig_marker,
                       ha="center", va="center",
                       color=text_color, fontsize=11,
                       fontweight='bold')

    # === LABELS AND TITLE ===
    ax.set_xlabel('Performance Metrics', fontweight='medium', fontsize=10)
    ax.set_ylabel('Agent Comparisons', fontweight='medium', fontsize=10)
    ax.set_title('Championship Tournament: Pairwise Agent Performance Comparison',
                fontweight='bold', fontsize=11, pad=15)

    # === ADD LEGEND FOR SIGNIFICANCE ===
    legend_text = '* p < 0.05   ** p < 0.01   *** p < 0.001'
    fig.text(0.5, 0.02, legend_text, ha='center', fontsize=9,
            style='italic', color='#555555',
            bbox=dict(boxstyle='round', facecolor='white', 
                     edgecolor='gray', alpha=0.8, linewidth=0.8))

    # === STYLING REFINEMENTS ===
    ax.spines['top'].set_linewidth(0.8)
    ax.spines['right'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.spines['left'].set_linewidth(0.8)

    # === SAVE FIGURE ===
    plt.tight_layout(rect=[0, 0.04, 1, 1])  # Leave space for legend at bottom
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {output_file}")

    # === PRINT STATISTICS ===
    print("\n" + "="*80)
    print("CHAMPIONSHIP TOURNAMENT EFFECT SIZE SUMMARY")
    print("="*80)

    print(f"\nOverall Statistics:")
    print(f"  Maximum effect size: {np.max(d_values):.3f}")
    print(f"  Minimum effect size: {np.min(d_values):.3f}")
    print(f"  Mean absolute effect size: {np.mean(np.abs(d_values)):.3f}")

    print(f"\nSignificant Effects (p < 0.05):")
    significant_count = np.sum(p_values < 0.05)
    total_comparisons = p_values.size
    print(f"  {significant_count}/{total_comparisons} comparisons ({100*significant_count/total_comparisons:.1f}%)")

    print(f"\nLargest Effect Sizes by Metric:")
    for j, metric in enumerate(metrics):
        max_idx = np.argmax(np.abs(d_values[:, j]))
        max_val = d_values[max_idx, j]
        max_comp = comparisons[max_idx]
        print(f"  {metric}: {max_val:.3f} ({max_comp})")

    print(f"\nAgressive Dominance Analysis:")
    aggressive_comparisons = d_values[0:3, :]  # First 3 rows
    print(f"  Mean effect size vs all opponents: {np.mean(aggressive_comparisons):.3f}")
    print(f"  Win rate advantage: {np.mean(aggressive_comparisons[:, 0]):.3f}")
    print(f"  Economic advantage: {np.mean(aggressive_comparisons[:, 1]):.3f}")
    print(f"  Jhyap advantage: {np.mean(aggressive_comparisons[:, 2]):.3f}")

    print(f"\nNon-Aggressive Comparisons:")
    non_aggressive = d_values[3:6, :]  # Last 3 rows
    print(f"  Mean effect size: {np.mean(np.abs(non_aggressive)):.3f}")
    print(f"  ISMCTS vs PPO (Win): {d_values[3, 0]:.3f}")
    print(f"  ISMCTS vs Random (Win): {d_values[4, 0]:.3f}")
    print(f"  PPO vs Random (Win): {d_values[5, 0]:.3f}")

    print("\n" + "="*80)

    plt.close()
    return True

def main():
    """Main execution function."""
    print("="*80)
    print("CHAMPIONSHIP TOURNAMENT HEATMAP - PUBLICATION QUALITY")
    print("="*80)

    print("\nGenerating championship tournament heatmap...")
    success = create_championship_heatmap()

    if success:
        print("\n" + "="*80)
        print("✓ SUCCESS: Championship heatmap generated!")
        print("="*80)
        print("\nVisualization features:")
        print("  • Diverging colormap (red = positive, blue = negative)")
        print("  • Cohen's d effect sizes displayed in cells")
        print("  • Statistical significance markers (*, **, ***)")
        print("  • Publication-ready formatting (300 DPI)")
        print("  • Aggressive agent dominance clearly visible")
        print("\nOutput: championship_comparison_heatmap.png")
        print("="*80)
    else:
        print("\n❌ FAILED: Could not generate heatmap")

if __name__ == "__main__":
    main()

CHAMPIONSHIP TOURNAMENT HEATMAP - PUBLICATION QUALITY

Generating championship tournament heatmap...
✓ Saved: championship_comparison_heatmap.png

CHAMPIONSHIP TOURNAMENT EFFECT SIZE SUMMARY

Overall Statistics:
  Maximum effect size: 5.567
  Minimum effect size: -0.076
  Mean absolute effect size: 1.976

Significant Effects (p < 0.05):
  14/18 comparisons (77.8%)

Largest Effect Sizes by Metric:
  Win: 3.613 (Aggressive vs Random)
  Economic: 2.759 (Aggressive vs PPO)
  Jhyap: 5.567 (Aggressive vs Random)

Agressive Dominance Analysis:
  Mean effect size vs all opponents: 3.697
  Win rate advantage: 3.265
  Economic advantage: 2.524
  Jhyap advantage: 5.303

Non-Aggressive Comparisons:
  Mean effect size: 0.255
  ISMCTS vs PPO (Win): 0.343
  ISMCTS vs Random (Win): 0.355
  PPO vs Random (Win): 0.017


✓ SUCCESS: Championship heatmap generated!

Visualization features:
  • Diverging colormap (red = positive, blue = negative)
  • Cohen's d effect sizes displayed in cells
  • Statistical